In [1]:
#Installing nltk
!pip install nltk

#Import Libraries
# Data handling
import pandas as pd
import numpy as np

# Text preprocessing
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, f1_score

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [5]:
# Load dataset
df = pd.read_csv("nlp_dataset (1).csv")

In [6]:
# Display first 5 rows
print(df.head())

print(df.info())

print(df['Emotion'].value_counts())

                                             Comment Emotion
0  i seriously hate one subject to death but now ...    fear
1                 im so full of life i feel appalled   anger
2  i sit here to write i start to dig out my feel...    fear
3  ive been really angry with r and i feel like a...     joy
4  i feel suspicious if there is no one outside l...    fear
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5937 entries, 0 to 5936
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Comment  5937 non-null   object
 1   Emotion  5937 non-null   object
dtypes: object(2)
memory usage: 92.9+ KB
None
Emotion
anger    2000
joy      2000
fear     1937
Name: count, dtype: int64


### Explanation

The dataset contains two columns:

* **Comment** → Input text data
* **Emotion** → Target label (emotion category)

The goal is to train machine learning models to predict emotions from text.

In [7]:
## Define Stopwords
stop_words = set(stopwords.words('english'))

In [8]:
## Create Cleaning Function

# Text cleaning function

def clean_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Join words back
    cleaned_text = ' '.join(tokens)

    return cleaned_text


In [9]:
## Apply Preprocessing

df['Cleaned_Comment'] = df['Comment'].apply(clean_text)

# Display cleaned data
print(df[['Comment', 'Cleaned_Comment']].head())

                                             Comment  \
0  i seriously hate one subject to death but now ...   
1                 im so full of life i feel appalled   
2  i sit here to write i start to dig out my feel...   
3  ive been really angry with r and i feel like a...   
4  i feel suspicious if there is no one outside l...   

                                     Cleaned_Comment  
0  seriously hate one subject death feel reluctan...  
1                         im full life feel appalled  
2  sit write start dig feelings think afraid acce...  
3  ive really angry r feel like idiot trusting fi...  
4  feel suspicious one outside like rapture happe...  


In [10]:
## Convert Text to Numerical Features

# Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer(max_features=5000)

# Transform text data
X = vectorizer.fit_transform(df['Cleaned_Comment'])

# Target labels
y = df['Emotion']

print(X.shape)

(5937, 5000)


In [11]:
# Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data size:", X_train.shape)
print("Testing data size:", X_test.shape)

Training data size: (4749, 5000)
Testing data size: (1188, 5000)


In [14]:
#Model Development

## Train Naive Bayes

# Initialize model
nb_model = MultinomialNB()

# Train model
nb_model.fit(X_train, y_train)

# Predictions
nb_predictions = nb_model.predict(X_test)

## Evaluate Naive Bayes

# Accuracy
nb_accuracy = accuracy_score(y_test, nb_predictions)

# F1 Score
nb_f1 = f1_score(y_test, nb_predictions, average='weighted')

print("Naive Bayes Accuracy:", nb_accuracy)
print("Naive Bayes F1 Score:", nb_f1)

print("\nClassification Report:\n")
print(classification_report(y_test, nb_predictions))

Naive Bayes Accuracy: 0.9090909090909091
Naive Bayes F1 Score: 0.9089193266783082

Classification Report:

              precision    recall  f1-score   support

       anger       0.88      0.95      0.91       392
        fear       0.91      0.92      0.92       416
         joy       0.94      0.86      0.90       380

    accuracy                           0.91      1188
   macro avg       0.91      0.91      0.91      1188
weighted avg       0.91      0.91      0.91      1188



In [15]:
# Support Vector Machine (SVM)

## Train SVM Model

# Initialize model
svm_model = LinearSVC()

# Train model
svm_model.fit(X_train, y_train)

# Predictions
svm_predictions = svm_model.predict(X_test)

## Evaluate SVM

# Accuracy
svm_accuracy = accuracy_score(y_test, svm_predictions)

# F1 Score
svm_f1 = f1_score(y_test, svm_predictions, average='weighted')

print("SVM Accuracy:", svm_accuracy)
print("SVM F1 Score:", svm_f1)

print("\nClassification Report:\n")
print(classification_report(y_test, svm_predictions))

SVM Accuracy: 0.952020202020202
SVM F1 Score: 0.9519770568978076

Classification Report:

              precision    recall  f1-score   support

       anger       0.94      0.96      0.95       392
        fear       0.97      0.93      0.95       416
         joy       0.95      0.97      0.96       380

    accuracy                           0.95      1188
   macro avg       0.95      0.95      0.95      1188
weighted avg       0.95      0.95      0.95      1188



In [16]:
# Compare both models
comparison = pd.DataFrame({
    'Model': ['Naive Bayes', 'Support Vector Machine'],
    'Accuracy': [nb_accuracy, svm_accuracy],
    'F1 Score': [nb_f1, svm_f1]
})

print(comparison)

                    Model  Accuracy  F1 Score
0             Naive Bayes  0.909091  0.908919
1  Support Vector Machine  0.952020  0.951977


# Explanation of Models

## Naive Bayes

Naive Bayes is a probabilistic machine learning algorithm based on Bayes’ theorem.

### Advantages

* Fast and efficient
* Works well with text classification
* Good for large datasets

### Suitability for Emotion Classification

Naive Bayes performs well for NLP tasks because it handles word frequency efficiently.

---

## Support Vector Machine (SVM)

SVM is a supervised machine learning algorithm that finds the optimal boundary between classes.

### Advantages

* High accuracy
* Performs well on high-dimensional text data
* Effective for classification problems

### Suitability for Emotion Classification

SVM is highly suitable for emotion classification because text data usually contains many features, and SVM handles sparse data effectively.

# Conclusion

In this project:

* Text data was cleaned and preprocessed
* TF-IDF was used for feature extraction
* Naive Bayes and SVM models were trained
* Models were evaluated using Accuracy and F1-score

### Final Observation

* SVM generally provides better performance for text classification tasks.
* TF-IDF improved the quality of features by assigning importance to meaningful words.